In [0]:
# Load the data from bronze table based on latest OrderDate (incremental load)
if spark.catalog.tableExists('practice.sales_silver.s_sales'):
    last_load_date = spark.sql("""
        SELECT MAX(OrderDate) 
        FROM practice.sales_silver.s_sales
    """).collect()[0][0]
    print('last_load_date:', last_load_date)
else:
    last_load_date = '1000-01-01'
    print('last_load_date:', last_load_date)

In [0]:
# Transformations
silver_df = spark.sql(f"""
    SELECT 
        *,
        ListPrice * Quantity AS Total_Sales,
        (ListPrice * Quantity) * (1 - DiscountPercent / 100) AS Final_Price,
        CURRENT_TIMESTAMP() AS processed_date       
    FROM practice.sales_bronze.b_sales
    WHERE OrderDate > '{last_load_date}'
""")

In [0]:
# Create the Silver table
silver_df.write.mode('append').saveAsTable('practice.sales_silver.s_sales')

In [0]:
%sql
-- View sample records from silver table
SELECT * 
FROM practice.sales_silver.s_sales 
ORDER BY OrderId DESC
LIMIT 5;